# Quick Start

Load NVD data → flatten → pandas. That's it.

In [ ]:
from vulnflow import VulnDataset
from vulnflow.transforms.flatten import (
    CPEPrimaryTransform,
    CVSSFlattenTransform,
    CWEPrimaryTransform,
)

## Load and flatten

NVD stores CVSS/CWE/CPE as nested JSON. Apply flatten transforms to get scalar columns.

In [35]:
ds = VulnDataset.from_nvd("../data/nvdcve-2.0-Modified.json").transform([
    CVSSFlattenTransform(),
    CWEPrimaryTransform(),
    CPEPrimaryTransform(),
])

df = ds.to_pandas()
df.head()

Transforming dataset: 100%|██████████| 3.00/3.00 [00:00<00:00, 87.8transforms/s, [CPEPrimaryTransform]]


,id,cve_id,aliases,title,description,first_seen_at,last_updated_at,sources,nvd_published_at,nvd_modified_at,...,cvss_scope,cvss_confidentiality,cvss_integrity,cvss_availability,cwe_id,cwe_name,cpe_vendor,cpe_product,cpe_version,cpe_fix_version
0,15c3ae39-bb01-4857-bc44-dc17ec691fe8,CVE-2012-2571,[],None,Multiple cross-site scripting (XSS) vulnerabil...,2026-02-04 16:19:52.323815,2026-02-04 16:19:52.323818,"[""nvd""]",2012-08-12 21:55:01.027,2026-01-27 15:53:34.823,...,NaN,NaN,NaN,NaN,CWE-79,None,winwebmail,winwebmail_server,3.8.1.6,NaN
1,376eb8a0-f71d-46c5-b4ff-56d89d8890fe,CVE-2015-1579,[],None,Directory traversal vulnerability in the Elega...,2026-02-04 16:19:52.323848,2026-02-04 16:19:52.323848,"[""nvd""]",2015-02-11 19:59:06.417,2026-01-28 20:39:35.730,...,NaN,NaN,NaN,NaN,CWE-22,None,elegantthemes,divi,-,NaN
2,52766819-9ab5-40f0-a420-46a042247ca1,CVE-2017-16539,[],None,The DefaultLinuxSpec function in oci/defaults....,2026-02-04 16:19:52.323886,2026-02-04 16:19:52.323887,"[""nvd""]",2017-11-04 17:29:00.207,2026-01-27 21:15:55.163,...,U,N,N,H,CWE-200,None,mobyproject,moby,*,NaN
3,19b89f89-e936-4acc-91f0-3f149a5f2833,CVE-2018-7543,[],None,Cross-site scripting (XSS) vulnerability in in...,2026-02-04 16:19:52.323911,2026-02-04 16:19:52.323911,"[""nvd""]",2018-03-26 18:29:01.440,2026-02-02 15:06:20.273,...,C,L,L,N,CWE-79,None,awesomemotive,duplicator,1.2.32,NaN
4,6a3a0ba7-6002-4650-aea5-9186ecfc6763,CVE-2018-17207,[],None,An issue was discovered in Snap Creek Duplicat...,2026-02-04 16:19:52.323932,2026-02-04 16:19:52.323932,"[""nvd""]",2018-09-19 16:29:01.223,2026-02-02 15:06:20.273,...,U,H,H,H,CWE-94,None,awesomemotive,duplicator,*,1.2.42


In [26]:
df.columns.tolist()

['id',
 'cve_id',
 'aliases',
 'title',
 'description',
 'first_seen_at',
 'last_updated_at',
 'sources',
 'nvd_published_at',
 'nvd_modified_at',
 'tags',
 'metadata',
 'nvd_status',
 'cvss_version',
 'cvss_base_score',
 'cvss_base_severity',
 'cvss_source',
 'cvss_vector_string',
 'cvss_exploitability_score',
 'cvss_impact_score',
 'cvss_attack_vector',
 'cvss_attack_complexity',
 'cvss_privileges_required',
 'cvss_user_interaction',
 'cvss_scope',
 'cvss_confidentiality',
 'cvss_integrity',
 'cvss_availability',
 'cwe_id',
 'cwe_name',
 'cpe_vendor',
 'cpe_product',
 'cpe_version',
 'cpe_fix_version']

## Critical vulnerabilities

In [30]:
critical = df[df['cvss_base_score'] >= 9.0]
critical[['cve_id', 'cvss_base_score', 'cwe_id', 'description']].head(10)

,cve_id,cvss_base_score,cwe_id,description
4,CVE-2018-17207,9.8,CWE-94,An issue was discovered in Snap Creek Duplicat...
8,CVE-2019-19006,9.8,CWE-287,"Sangoma FreePBX 115.0.16.26 and below, 14.0.13..."
23,CVE-2023-26813,9.8,CWE-89,SQL injection vulnerability in com.xnx3.wangma...
49,CVE-2018-25095,9.8,NaN,The Duplicator WordPress plugin before 1.3.0 d...
53,CVE-2024-25220,9.8,CWE-89,Task Manager App v1.0 was discovered to contai...
55,CVE-2024-25222,9.8,CWE-89,Task Manager App v1.0 was discovered to contai...
66,CVE-2024-30264,9.3,CWE-79,Typebot is an open-source chatbot builder. A r...
70,CVE-2024-2420,9.8,CWE-259,LenelS2 NetBox access control and event monito...
71,CVE-2024-2421,9.8,CWE-78,LenelS2 NetBox access control and event monito...
74,CVE-2024-37282,9.8,CWE-285,It was identified that under certain specific ...


## Severity distribution

In [31]:
df['cvss_base_severity'].value_counts()

cvss_base_severity
MEDIUM      1022
HIGH         913
CRITICAL     316
LOW           57
NONE           1
Name: count, dtype: int64

## Top CWEs

In [32]:
df['cwe_id'].value_counts().head(10)

cwe_id
CWE-79     424
CWE-862    127
CWE-89     102
CWE-22      86
CWE-98      79
CWE-352     61
CWE-428     61
CWE-78      58
CWE-20      56
CWE-284     50
Name: count, dtype: int64

## Cache to parquet

~10x faster to load than JSON.

In [33]:
ds.to_parquet("../data/cache.parquet")

In [34]:
# Reload instantly
df = VulnDataset.from_parquet("../data/cache.parquet").to_pandas()
len(df)

2767